# Export báo cáo bác sĩ cho local demo

Notebook này chỉ lấy trường `Report` từ `train.csv` cho đúng một `StudyInstanceUID`, sau đó tạo file văn bản `radiology_report.txt`. **Không tải lại DICOM, không export 12 ground-truth labels và không chạy model.**

`Report` là văn bản báo cáo gốc trong dữ liệu train. Báo cáo có thể dùng ngôn ngữ khác tiếng Việt, vì vậy notebook giữ nguyên văn, không tự dịch hay sửa nội dung.

In [ ]:
from pathlib import Path
import os
import zipfile
import pandas as pd
from IPython.display import display, FileLink

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
COMP_ROOT_OVERRIDE = ""  # Điền nếu Kaggle mount train.csv ở vị trí khác.

# StudyInstanceUID hiện tại lấy từ thư mục demo_sample mà bạn đã export.
# Chỉ thay đổi nếu manifest.json của sample hiện tại có UID khác.
SAMPLE_STUDY_UID = "1.2.826.0.1.3680043.8.498.10095687747295410396510538520594649149"

OUTPUT_DIR = WORK_ROOT / "rsna_demo_clinical_context"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_PATH = OUTPUT_DIR / "radiology_report.txt"
OUTPUT_ZIP = WORK_ROOT / "demo_radiology_report.zip"

print("Input root:", INPUT_ROOT)
print("Study UID:", SAMPLE_STUDY_UID)

In [ ]:
def discover_train_csv() -> Path:
    if COMP_ROOT_OVERRIDE:
        candidate = Path(COMP_ROOT_OVERRIDE).expanduser() / "train.csv"
        if not candidate.is_file():
            raise FileNotFoundError(candidate)
        return candidate

    direct_candidates = [
        INPUT_ROOT / "competitions" / "rsna-knee-abnormality-detection" / "train.csv",
        INPUT_ROOT / "rsna-knee-abnormality-detection" / "train.csv",
    ]
    for candidate in direct_candidates:
        if candidate.is_file():
            return candidate

    skip_dirs = {"train_series", "test_series", "train_images", "test_images", "__pycache__"}
    matches = []
    for current, dirs, files in os.walk(INPUT_ROOT):
        dirs[:] = [name for name in dirs if name.lower() not in skip_dirs]
        if "train.csv" in files:
            matches.append(Path(current) / "train.csv")
    unique = {str(path.resolve()): path for path in matches}
    if len(unique) != 1:
        raise RuntimeError(
            "Không tìm thấy duy nhất train.csv. "
            f"Các file tìm được: {list(unique.values())}. "
            "Hãy điền COMP_ROOT_OVERRIDE ở Cell 1."
        )
    return next(iter(unique.values()))

TRAIN_CSV = discover_train_csv()
print("Train CSV:", TRAIN_CSV)

# pandas.read_csv is required here because the Report field may contain
# embedded newlines; reading the file line-by-line would corrupt row matching.
try:
    train = pd.read_csv(TRAIN_CSV, dtype={"StudyInstanceUID": str}, encoding="utf-8-sig")
except UnicodeDecodeError:
    train = pd.read_csv(TRAIN_CSV, dtype={"StudyInstanceUID": str}, encoding="cp1252")

required = {"StudyInstanceUID", "Report"}
missing = required - set(train.columns)
if missing:
    raise ValueError(f"train.csv thiếu cột bắt buộc: {sorted(missing)}")

rows = train.loc[train["StudyInstanceUID"].astype(str).eq(SAMPLE_STUDY_UID)]
if len(rows) != 1:
    raise ValueError(
        f"Cần đúng một record cho StudyInstanceUID={SAMPLE_STUDY_UID}, tìm thấy {len(rows)}. "
        "Kiểm tra lại UID trong manifest.json."
    )

report_value = rows.iloc[0]["Report"]
if pd.isna(report_value) or not str(report_value).strip():
    raise ValueError("Study này không có nội dung Report trong train.csv.")
report_text = str(report_value).strip()
print("Report characters:", len(report_text))
print("Report preview:")
print(report_text[:1200])

In [ ]:
# The header lets the local UI verify that this report belongs to the opened DICOM study.
# The report body itself is preserved verbatim from train.csv.
content = (
    f"StudyInstanceUID: {SAMPLE_STUDY_UID}\n"
    "Source: RSNA train.csv / Report\n"
    "\n"
    "----------------------------------------\n"
    f"{report_text}\n"
)
REPORT_PATH.write_text(content, encoding="utf-8")

if OUTPUT_ZIP.exists():
    OUTPUT_ZIP.unlink()
with zipfile.ZipFile(OUTPUT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(REPORT_PATH, arcname="radiology_report.txt")

with zipfile.ZipFile(OUTPUT_ZIP, "r") as archive:
    bad_member = archive.testzip()
    if bad_member is not None:
        raise RuntimeError(f"ZIP integrity test failed: {bad_member}")
    assert "radiology_report.txt" in archive.namelist()

print("\nEXPORT COMPLETE")
print("Text file:", REPORT_PATH)
print("ZIP:", OUTPUT_ZIP)
display(FileLink(str(OUTPUT_ZIP)))

## Cách tích hợp vào project local

1. Tải `demo_radiology_report.zip` từ output của notebook.
2. Giải nén file `radiology_report.txt`.
3. Copy file đó vào **cùng thư mục với** `manifest.json` của sample hiện tại, ví dụ `C:\rsna_demo\demo_sample\radiology_report.txt`.
4. Khởi động lại dashboard. UI sẽ xác thực `StudyInstanceUID` ghi trong file trước khi hiển thị.

Notebook không chứa hoặc export 12 nhãn ground-truth. Nội dung báo cáo được giữ nguyên ngôn ngữ gốc; không phải bản dịch hay kết luận do mô hình tạo ra.